# Supplier Compliance Intelligence — 01 Data Validation & KPI Reconciliation

**Purpose:** check that the source workbook is structurally sound, that the table relationships hold, that the
illustrative risk model can be reproduced, and that every reference KPI in `KPI_Summary` can be independently
recalculated from the underlying data.

**Rules followed in this notebook**
- The workbook `data/Supplier_Compliance_Intelligence.xlsx` is opened **read-only**. Nothing is written back to it,
  and its SHA-256 hash is checked at the start and end.
- Every check is recorded in a `checks` list; the final cell fails loudly if any check does not pass.
- Reporting cutoff: **30 September 2026**.

**Interpretation reminders** (from the workbook's START HERE sheet)
- Submission ≠ completion ≠ confirmed regulatory compliance.
- The risk score is an *illustrative prioritisation model*, not a validated legal risk rating.
- The data is simulated educational data — not real supplier records.

Run with: *Kernel → Restart & Run All* (kernel: the project's `.venv`).

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORKBOOK = PROJECT_ROOT / "data" / "Supplier_Compliance_Intelligence.xlsx"
CUTOFF = pd.Timestamp("2026-09-30")

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

HASH_AT_START = sha256(WORKBOOK)
checks = []  # (area, description, passed, detail)

def check(area, description, passed, detail=""):
    checks.append((area, description, bool(passed), detail))
    print(f"[{'PASS' if passed else 'FAIL'}] {area}: {description} {detail}")

print("Workbook:", WORKBOOK.name)
print("SHA-256 :", HASH_AT_START)

Workbook: Supplier_Compliance_Intelligence.xlsx
SHA-256 : b26ad0279b30265d621b89113272becb191bb73b1976b1f33da6fe0d8583e601


## 1. Load the workbook
All sheets are loaded into memory. Dates in the source are stored as **text** (`YYYY-MM-DD`), so they are converted
to proper dates here. Tableau will need the same conversion (change the field type to *Date*).

In [2]:
sheets = pd.read_excel(WORKBOOK, sheet_name=None)

sm = sheets["Supplier_Master"].copy()
cr = sheets["Compliance_Requests"].copy()
se = sheets["Supplier_Engagement"].copy()
ra = sheets["Risk_Assessment"].copy()
kpi_ref = sheets["KPI_Summary"].copy()

date_cols = {
    "cr": (cr, ["Campaign_Date", "Due_Date", "Submission_Date"]),
    "se": (se, ["Follow_Up_Date"]),
    "ra": (ra, ["Assessment_Date"]),
}
for name, (df, cols) in date_cols.items():
    for c in cols:
        parsed = pd.to_datetime(df[c], format="%Y-%m-%d", errors="coerce")
        bad = (df[c].notna() & parsed.isna()).sum()
        check("Dates", f"{c} text values all parse as YYYY-MM-DD", bad == 0, f"(unparseable: {bad})")
        df[c] = parsed

pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in sheets.items()],
    columns=["Sheet", "Rows (excl. header)", "Columns"],
)

[PASS] Dates: Campaign_Date text values all parse as YYYY-MM-DD (unparseable: 0)
[PASS] Dates: Due_Date text values all parse as YYYY-MM-DD (unparseable: 0)
[PASS] Dates: Submission_Date text values all parse as YYYY-MM-DD (unparseable: 0)
[PASS] Dates: Follow_Up_Date text values all parse as YYYY-MM-DD (unparseable: 0)
[PASS] Dates: Assessment_Date text values all parse as YYYY-MM-DD (unparseable: 0)


,Sheet,Rows (excl. header),Columns
0,START HERE,11,2
1,Supplier_Master,800,7
2,Compliance_Requests,2139,11
3,Supplier_Engagement,1499,7
4,Risk_Assessment,800,11
5,KPI_Summary,11,2
6,Risk_Weights,4,3
7,Data_Dictionary,36,3


## 2. Column profile of the four data tables
For each column: data type, missing values, number of distinct values, and range (numbers/dates) or top values.

In [3]:
def profile(df):
    rows = []
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s):
            summary = f"min={s.min()}  max={s.max()}"
        else:
            summary = ", ".join(f"{k}={v}" for k, v in s.value_counts().head(5).items())
        rows.append((c, str(s.dtype), int(s.isna().sum()), int(s.nunique()), summary))
    return pd.DataFrame(rows, columns=["Column", "Type", "Missing", "Distinct", "Range / top values"])

for name, df in [("Supplier_Master", sm), ("Compliance_Requests", cr),
                 ("Supplier_Engagement", se), ("Risk_Assessment", ra)]:
    print(f"\n=== {name}  {df.shape[0]} rows x {df.shape[1]} cols   full-row duplicates: {df.duplicated().sum()}")
    display(profile(df))


=== Supplier_Master  800 rows x 7 cols   full-row duplicates: 0


,Column,Type,Missing,Distinct,Range / top values
0,Supplier_ID,str,0,800,"SUP-0001=1, SUP-0002=1, SUP-0003=1, SUP-0004=1..."
1,Supplier_Name,str,0,800,"Simulated Supplier 0001=1, Simulated Supplier ..."
2,Region,str,0,4,"Asia-Pacific=279, Europe=241, Americas=207, Af..."
3,Industry_Category,str,0,5,"Electronics=222, Automotive=208, Industrial Co..."
4,Supplier_Criticality,str,0,3,"Medium=379, Low=245, High=176"
5,Active_Part_Count,int64,0,398,min=3 max=847
6,Supplier_Status,str,0,1,Active=800



=== Compliance_Requests  2139 rows x 11 cols   full-row duplicates: 0


,Column,Type,Missing,Distinct,Range / top values
0,Request_ID,str,0,2139,"REQ-00001=1, REQ-00002=1, REQ-00003=1, REQ-000..."
1,Supplier_ID,str,0,800,"SUP-0003=4, SUP-0005=4, SUP-0016=4, SUP-0021=4..."
2,Compliance_Module,str,0,4,"RoHS=543, REACH=538, PFAS=537, CMRT=521"
3,Campaign_Date,datetime64[us],0,316,min=2025-10-01 00:00:00 max=2026-08-12 00:00:00
4,Due_Date,datetime64[us],0,316,min=2025-11-15 00:00:00 max=2026-09-26 00:00:00
5,Submission_Date,datetime64[us],612,337,min=2025-10-19 00:00:00 max=2026-09-30 00:00:00
6,Response_Status,str,0,4,"Complete=1193, Pending=581, Incomplete=281, De..."
7,Compliance_Outcome,str,0,4,"Not assessed=946, Compliant=941, Unknown=151, ..."
8,Overdue_Days,int64,0,305,min=0 max=319
9,Submitted_Flag,int64,0,2,min=0 max=1



=== Supplier_Engagement  1499 rows x 7 cols   full-row duplicates: 0


,Column,Type,Missing,Distinct,Range / top values
0,Engagement_ID,str,0,1499,"ENG-00001=1, ENG-00002=1, ENG-00003=1, ENG-000..."
1,Request_ID,str,0,990,"REQ-00045=4, REQ-00068=4, REQ-00216=4, REQ-002..."
2,Supplier_ID,str,0,629,"SUP-0565=9, SUP-0080=8, SUP-0090=8, SUP-0637=8..."
3,Follow_Up_Date,datetime64[us],0,311,min=2025-11-22 00:00:00 max=2026-09-30 00:00:00
4,Follow_Up_Channel,str,0,3,"Email=1127, Phone=255, Escalation=117"
5,Engagement_Outcome,str,0,2,"No confirmed response=944, Responded later=555"
6,Assigned_Team,str,0,3,"Compliance Operations=980, Supplier Relations=..."



=== Risk_Assessment  800 rows x 11 cols   full-row duplicates: 0


,Column,Type,Missing,Distinct,Range / top values
0,Supplier_ID,str,0,800,"SUP-0001=1, SUP-0002=1, SUP-0003=1, SUP-0004=1..."
1,Assessment_Date,datetime64[us],0,1,min=2026-09-30 00:00:00 max=2026-09-30 00:00:00
2,Criticality_Score,int64,0,3,min=20 max=100
3,Information_Gap_Score,float64,0,7,min=0.0 max=100.0
4,Overdue_Severity_Score,float64,0,81,min=0.0 max=100.0
5,Business_Exposure_Score,float64,0,398,min=0.35 max=99.65
6,Overall_Risk_Score,float64,0,713,min=7.05 max=99.89
7,Risk_Category,str,0,3,"Medium=366, Low=253, High=181"
8,Eligible_Request_Count,int64,0,4,min=1 max=4
9,Information_Gap_Count,int64,0,5,min=0 max=4


## 3. Identifiers (primary keys) and grain
| Table | Expected key | Grain |
|---|---|---|
| Supplier_Master | Supplier_ID | one row per supplier |
| Compliance_Requests | Request_ID | one row per supplier × compliance module |
| Supplier_Engagement | Engagement_ID | one row per follow-up event |
| Risk_Assessment | Supplier_ID | one row per supplier |

In [4]:
for name, df, key in [("Supplier_Master", sm, "Supplier_ID"),
                      ("Compliance_Requests", cr, "Request_ID"),
                      ("Supplier_Engagement", se, "Engagement_ID"),
                      ("Risk_Assessment", ra, "Supplier_ID")]:
    check("Keys", f"{name}.{key} is unique and non-null",
          df[key].is_unique and df[key].notna().all(), f"({df[key].nunique()} distinct / {len(df)} rows)")
    check("Duplicates", f"{name} has no full-row duplicates", df.duplicated().sum() == 0)

check("Grain", "Compliance_Requests has one row per Supplier_ID x Compliance_Module",
      not cr.duplicated(["Supplier_ID", "Compliance_Module"]).any())

[PASS] Keys: Supplier_Master.Supplier_ID is unique and non-null (800 distinct / 800 rows)
[PASS] Duplicates: Supplier_Master has no full-row duplicates 
[PASS] Keys: Compliance_Requests.Request_ID is unique and non-null (2139 distinct / 2139 rows)
[PASS] Duplicates: Compliance_Requests has no full-row duplicates 
[PASS] Keys: Supplier_Engagement.Engagement_ID is unique and non-null (1499 distinct / 1499 rows)
[PASS] Duplicates: Supplier_Engagement has no full-row duplicates 
[PASS] Keys: Risk_Assessment.Supplier_ID is unique and non-null (800 distinct / 800 rows)
[PASS] Duplicates: Risk_Assessment has no full-row duplicates 
[PASS] Grain: Compliance_Requests has one row per Supplier_ID x Compliance_Module 


## 4. Relationships
Expected (from START HERE):
1. `Supplier_Master.Supplier_ID → Compliance_Requests.Supplier_ID`
2. `Supplier_Master.Supplier_ID → Risk_Assessment.Supplier_ID`
3. `Compliance_Requests.Request_ID → Supplier_Engagement.Request_ID`

We check for orphans in both directions and show why a **physical join** would be wrong in Tableau.

In [5]:
suppliers = set(sm.Supplier_ID)

check("Relationships", "Every request's Supplier_ID exists in Supplier_Master", cr.Supplier_ID.isin(suppliers).all())
check("Relationships", "Every supplier has at least one request", suppliers <= set(cr.Supplier_ID))
check("Relationships", "Every Risk_Assessment Supplier_ID exists in Supplier_Master", ra.Supplier_ID.isin(suppliers).all())
check("Relationships", "Every supplier has a risk assessment", suppliers <= set(ra.Supplier_ID))
check("Relationships", "Every engagement's Request_ID exists in Compliance_Requests", se.Request_ID.isin(cr.Request_ID).all())

owner = se.merge(cr[["Request_ID", "Supplier_ID"]], on="Request_ID", suffixes=("", "_request"))
check("Relationships", "Engagement Supplier_ID matches the supplier on the linked request",
      (owner.Supplier_ID == owner.Supplier_ID_request).all())

print("\nRequests per supplier:", cr.Supplier_ID.value_counts().value_counts().sort_index().to_dict())
print("Engagements per followed-up request:", se.Request_ID.value_counts().value_counts().sort_index().to_dict())
print("Requests with no follow-up:", (~cr.Request_ID.isin(se.Request_ID)).sum())

joined_rows = len(cr.merge(se, on="Request_ID", how="left"))
print(f"\nPhysical LEFT JOIN requests x engagements: {joined_rows} rows vs {len(cr)} requests "
      f"-> {joined_rows - len(cr)} duplicated request rows. Use Tableau logical relationships instead.")

[PASS] Relationships: Every request's Supplier_ID exists in Supplier_Master 
[PASS] Relationships: Every supplier has at least one request 
[PASS] Relationships: Every Risk_Assessment Supplier_ID exists in Supplier_Master 
[PASS] Relationships: Every supplier has a risk assessment 
[PASS] Relationships: Every engagement's Request_ID exists in Compliance_Requests 
[PASS] Relationships: Engagement Supplier_ID matches the supplier on the linked request 

Requests per supplier: {1: 98, 2: 235, 3: 297, 4: 170}
Engagements per followed-up request: {1: 643, 2: 216, 3: 100, 4: 31}
Requests with no follow-up: 1149

Physical LEFT JOIN requests x engagements: 2648 rows vs 2139 requests -> 509 duplicated request rows. Use Tableau logical relationships instead.


## 5. Field-level business rules
These confirm that the pre-computed flags and day counts mean what the Data Dictionary says they mean.

In [6]:
check("Rules", "Submitted_Flag = 1 exactly when Submission_Date exists",
      (cr.Submission_Date.notna().astype(int) == cr.Submitted_Flag).all())
check("Rules", "Complete_Flag = 1 exactly when Response_Status = 'Complete'",
      ((cr.Response_Status == "Complete").astype(int) == cr.Complete_Flag).all())
check("Rules", "Due_Date is always Campaign_Date + 45 days",
      ((cr.Due_Date - cr.Campaign_Date).dt.days == 45).all())
check("Rules", "No submission before its campaign date", not (cr.Submission_Date < cr.Campaign_Date).any())
check("Rules", "No submission after the cutoff", not (cr.Submission_Date > CUTOFF).any())

expected_overdue = np.where(cr.Complete_Flag == 0, (CUTOFF - cr.Due_Date).dt.days.clip(lower=0), 0)
check("Rules", "Overdue_Days = max(0, cutoff - due date) for non-complete requests, else 0",
      (expected_overdue == cr.Overdue_Days).all())

fu = se.merge(cr[["Request_ID", "Campaign_Date"]], on="Request_ID")
check("Rules", "Follow-ups fall between campaign date and cutoff",
      ((fu.Follow_Up_Date >= fu.Campaign_Date) & (fu.Follow_Up_Date <= CUTOFF)).all())
check("Rules", "Risk_Assessment dated at the cutoff", (ra.Assessment_Date == CUTOFF).all())

print("\nResponse status vs submission:")
display(pd.crosstab(cr.Response_Status, cr.Submitted_Flag, margins=True))
print("Response status vs compliance outcome (outcome only assessed for Complete requests):")
display(pd.crosstab(cr.Response_Status, cr.Compliance_Outcome, margins=True))

[PASS] Rules: Submitted_Flag = 1 exactly when Submission_Date exists 
[PASS] Rules: Complete_Flag = 1 exactly when Response_Status = 'Complete' 
[PASS] Rules: Due_Date is always Campaign_Date + 45 days 
[PASS] Rules: No submission before its campaign date 
[PASS] Rules: No submission after the cutoff 
[PASS] Rules: Overdue_Days = max(0, cutoff - due date) for non-complete requests, else 0 
[PASS] Rules: Follow-ups fall between campaign date and cutoff 
[PASS] Rules: Risk_Assessment dated at the cutoff 

Response status vs submission:


Submitted_Flag,0,1,All
Response_Status,,,
Complete,0,1193,1193
Declined,31,53,84
Incomplete,0,281,281
Pending,581,0,581
All,612,1527,2139


Response status vs compliance outcome (outcome only assessed for Complete requests):


Compliance_Outcome,Compliant,Non-compliant,Not assessed,Unknown,All
Response_Status,,,,,
Complete,941,101,0,151,1193
Declined,0,0,84,0,84
Incomplete,0,0,281,0,281
Pending,0,0,581,0,581
All,941,101,946,151,2139


## 6. Reproduce the illustrative risk model
From `Risk_Weights`:

| Component | Weight | Definition |
|---|---|---|
| Criticality | 0.35 | High = 100, Medium = 60, Low = 20 |
| Information gap | 0.30 | 100 × non-complete requests / eligible requests |
| Overdue severity | 0.20 | min(100, max overdue days / 120 × 100) |
| Business exposure | 0.15 | min(100, active parts / 850 × 100) |

Category: High ≥ 70; Medium ≥ 40 and < 70; Low < 40. The source scores are rounded to 2 decimals, so a tolerance of
0.01 is used.

In [7]:
TOL = 0.01
per_supplier = cr.groupby("Supplier_ID").agg(
    eligible=("Request_ID", "size"),
    gap=("Complete_Flag", lambda s: int((s == 0).sum())),
    max_overdue=("Overdue_Days", "max"),
).reset_index()

r = ra.merge(sm, on="Supplier_ID").merge(per_supplier, on="Supplier_ID")
r["calc_crit"] = r.Supplier_Criticality.map({"High": 100, "Medium": 60, "Low": 20})
r["calc_gap"] = 100 * r.gap / r.eligible
r["calc_overdue"] = np.minimum(100, r.max_overdue / 120 * 100)
r["calc_exposure"] = np.minimum(100, r.Active_Part_Count / 850 * 100)
r["calc_overall"] = (0.35 * r.calc_crit + 0.30 * r.calc_gap
                     + 0.20 * r.calc_overdue + 0.15 * r.calc_exposure)
r["calc_category"] = np.select([r.Overall_Risk_Score >= 70, r.Overall_Risk_Score >= 40], ["High", "Medium"], "Low")

check("Risk", "Eligible_Request_Count matches request count", (r.eligible == r.Eligible_Request_Count).all())
check("Risk", "Information_Gap_Count matches non-complete count", (r.gap == r.Information_Gap_Count).all())
check("Risk", "Maximum_Overdue_Days matches max Overdue_Days", (r.max_overdue == r.Maximum_Overdue_Days).all())
for calc, src in [("calc_crit", "Criticality_Score"), ("calc_gap", "Information_Gap_Score"),
                  ("calc_overdue", "Overdue_Severity_Score"), ("calc_exposure", "Business_Exposure_Score"),
                  ("calc_overall", "Overall_Risk_Score")]:
    diff = (r[calc] - r[src]).abs().max()
    check("Risk", f"{src} reproduces within {TOL}", diff <= TOL, f"(max abs diff {diff:.4f})")
check("Risk", "Risk_Category matches thresholds", (r.calc_category == r.Risk_Category).all())

near = r.loc[((r.Overall_Risk_Score - 70).abs() < 0.5) | ((r.Overall_Risk_Score - 40).abs() < 0.5),
             ["Supplier_ID", "Overall_Risk_Score", "Risk_Category"]].sort_values("Overall_Risk_Score")
print(f"\nSuppliers within 0.5 points of a category threshold ({len(near)}) — category is sensitive here:")
display(near)

[PASS] Risk: Eligible_Request_Count matches request count 


[PASS] Risk: Information_Gap_Count matches non-complete count 
[PASS] Risk: Maximum_Overdue_Days matches max Overdue_Days 
[PASS] Risk: Criticality_Score reproduces within 0.01 (max abs diff 0.0000)
[PASS] Risk: Information_Gap_Score reproduces within 0.01 (max abs diff 0.0033)
[PASS] Risk: Overdue_Severity_Score reproduces within 0.01 (max abs diff 0.0033)
[PASS] Risk: Business_Exposure_Score reproduces within 0.01 (max abs diff 0.0047)
[PASS] Risk: Overall_Risk_Score reproduces within 0.01 (max abs diff 0.0067)
[PASS] Risk: Risk_Category matches thresholds 

Suppliers within 0.5 points of a category threshold (2) — category is sensitive here:


,Supplier_ID,Overall_Risk_Score,Risk_Category
609,SUP-0610,70.01,High
204,SUP-0205,70.12,High


## 7. KPI reconciliation
Each KPI is recalculated from the underlying tables and compared to `KPI_Summary`. **Grain** says which table/level
the KPI is counted at. Rates are compared at 4 decimal places; average response time at 1 decimal place (as supplied).

In [8]:
response_days = (cr.Submission_Date - cr.Campaign_Date).dt.days
followed = se.groupby("Request_ID").Engagement_Outcome.apply(lambda s: (s == "Responded later").any())

kpis = [
    ("Active suppliers", "Supplier", "count of suppliers with Supplier_Status = 'Active'",
     (sm.Supplier_Status == "Active").sum()),
    ("Eligible requests", "Request", "count of rows in Compliance_Requests", len(cr)),
    ("Submitted requests", "Request", "sum of Submitted_Flag", cr.Submitted_Flag.sum()),
    ("Request response rate", "Request", "submitted requests / eligible requests", cr.Submitted_Flag.mean()),
    ("Request completion rate", "Request", "complete requests / eligible requests", cr.Complete_Flag.mean()),
    ("Supplier response rate", "Supplier", "suppliers with >= 1 submitted request / all suppliers",
     cr.groupby("Supplier_ID").Submitted_Flag.max().mean()),
    ("Overdue backlog", "Request", "requests with Overdue_Days > 0", (cr.Overdue_Days > 0).sum()),
    ("Average response time (days)", "Request (submitted only)",
     "mean(Submission_Date - Campaign_Date) over submitted requests", response_days.mean()),
    ("High-priority suppliers", "Supplier", "suppliers with Risk_Category = 'High'", (ra.Risk_Category == "High").sum()),
    ("Requests with follow-up", "Request", "distinct Request_ID in Supplier_Engagement", se.Request_ID.nunique()),
    ("Follow-up conversion rate", "Request (followed-up only)",
     "followed-up requests with any 'Responded later' event / requests with follow-up", followed.mean()),
]
recon = pd.DataFrame(kpis, columns=["Metric", "Grain", "Formula", "Calculated"])
check("KPI", "Every KPI_Summary metric has a recalculation (and vice versa)",
      set(recon.Metric) == set(kpi_ref.Metric))
recon = kpi_ref.rename(columns={"Value": "Reference"}).merge(recon, on="Metric", how="left")
recon = recon[["Metric", "Grain", "Formula", "Calculated", "Reference"]]

decimals = {"Average response time (days)": 1}
recon["Match"] = [round(float(c), decimals.get(m, 4)) == round(float(ref), decimals.get(m, 4))
                  for m, c, ref in zip(recon.Metric, recon.Calculated, recon.Reference)]
for m, ok, c, ref in zip(recon.Metric, recon.Match, recon.Calculated, recon.Reference):
    check("KPI", f"{m} reconciles", ok, f"(calc {c:.4f} vs ref {ref:.4f})")
recon

[PASS] KPI: Every KPI_Summary metric has a recalculation (and vice versa) 
[PASS] KPI: Active suppliers reconciles (calc 800.0000 vs ref 800.0000)
[PASS] KPI: Eligible requests reconciles (calc 2139.0000 vs ref 2139.0000)
[PASS] KPI: Submitted requests reconciles (calc 1527.0000 vs ref 1527.0000)
[PASS] KPI: Request response rate reconciles (calc 0.7139 vs ref 0.7139)
[PASS] KPI: Request completion rate reconciles (calc 0.5577 vs ref 0.5577)
[PASS] KPI: Supplier response rate reconciles (calc 0.9225 vs ref 0.9225)
[PASS] KPI: Overdue backlog reconciles (calc 946.0000 vs ref 946.0000)
[PASS] KPI: Average response time (days) reconciles (calc 55.1434 vs ref 55.1000)
[PASS] KPI: High-priority suppliers reconciles (calc 181.0000 vs ref 181.0000)
[PASS] KPI: Requests with follow-up reconciles (calc 990.0000 vs ref 990.0000)
[PASS] KPI: Follow-up conversion rate reconciles (calc 0.4859 vs ref 0.4859)


,Metric,Grain,Formula,Calculated,Reference,Match
0,Active suppliers,Supplier,count of suppliers with Supplier_Status = 'Act...,800.000000,800.000000,True
1,Eligible requests,Request,count of rows in Compliance_Requests,2139.000000,2139.000000,True
2,Submitted requests,Request,sum of Submitted_Flag,1527.000000,1527.000000,True
3,Request response rate,Request,submitted requests / eligible requests,0.713885,0.713885,True
4,Request completion rate,Request,complete requests / eligible requests,0.557737,0.557737,True
5,Supplier response rate,Supplier,suppliers with >= 1 submitted request / all su...,0.922500,0.922500,True
6,Overdue backlog,Request,requests with Overdue_Days > 0,946.000000,946.000000,True
7,Average response time (days),Request (submitted only),mean(Submission_Date - Campaign_Date) over sub...,55.143418,55.100000,True
8,High-priority suppliers,Supplier,suppliers with Risk_Category = 'High',181.000000,181.000000,True
9,Requests with follow-up,Request,distinct Request_ID in Supplier_Engagement,990.000000,990.000000,True


## 8. Interpretation caveats (alternative definitions)
These are **not** discrepancies — the reference values reconcile above — but they show how the same words could be
read differently. Use the reference definitions in Tableau and wording that avoids the alternatives.

In [9]:
not_submitted_past_due = ((cr.Submitted_Flag == 0) & (cr.Due_Date < CUTOFF)).sum()
alt = pd.DataFrame([
    ("Overdue backlog", (cr.Overdue_Days > 0).sum(),
     "Non-complete requests (incl. submitted-but-incomplete and declined)"),
    ("Overdue backlog — alt.", not_submitted_past_due, "Only requests never submitted and past due"),
    ("Overdue = not complete?", (cr.Overdue_Days > 0).sum() == (cr.Complete_Flag == 0).sum(),
     f"Latest due date is {cr.Due_Date.max().date()}, before the cutoff, so every non-complete request is overdue"),
    ("Avg response time — median", response_days.median(), "Median days, campaign -> submission"),
    ("Avg days after due date", (cr.Submission_Date - cr.Due_Date).dt.days.mean(),
     "Average submission is this many days AFTER the 45-day due date"),
    ("Follow-up conversion — event level", (se.Engagement_Outcome == "Responded later").mean(),
     "Share of engagement events (not requests) with 'Responded later'"),
    ("Complete & declared Compliant", ((cr.Complete_Flag == 1) & (cr.Compliance_Outcome == "Compliant")).sum(),
     "Of 1,193 complete requests; outcome is self-declared, not independently verified"),
], columns=["Measure", "Value", "Meaning"])
display(alt)

print("Followed-up requests: 'Responded later' vs eventually submitted")
display(pd.crosstab(followed.rename("Any 'Responded later'"),
                    cr.set_index("Request_ID").loc[followed.index, "Submitted_Flag"]))
print("Association only — this does not show that follow-up CAUSED the response.")

,Measure,Value,Meaning
0,Overdue backlog,946,Non-complete requests (incl. submitted-but-inc...
1,Overdue backlog — alt.,612,Only requests never submitted and past due
2,Overdue = not complete?,True,"Latest due date is 2026-09-26, before the cuto..."
3,Avg response time — median,55.0,"Median days, campaign -> submission"
4,Avg days after due date,10.143418,Average submission is this many days AFTER the...
5,Follow-up conversion — event level,0.370247,Share of engagement events (not requests) with...
6,Complete & declared Compliant,941,"Of 1,193 complete requests; outcome is self-de..."


Followed-up requests: 'Responded later' vs eventually submitted


Submitted_Flag,0,1
Any 'Responded later',,
False,509,0
True,0,481


Association only — this does not show that follow-up CAUSED the response.


## 9. Summary of checks

In [10]:
summary = pd.DataFrame(checks, columns=["Area", "Check", "Passed", "Detail"])
check("Read-only", "Workbook unchanged (SHA-256 identical to start)", sha256(WORKBOOK) == HASH_AT_START)
summary = pd.DataFrame(checks, columns=["Area", "Check", "Passed", "Detail"])
print(f"{summary.Passed.sum()} / {len(summary)} checks passed")
display(summary.groupby("Area").Passed.agg(["sum", "count"]))
failed = summary[~summary.Passed]
assert failed.empty, f"Failed checks:\n{failed.to_string()}"

[PASS] Read-only: Workbook unchanged (SHA-256 identical to start) 
50 / 50 checks passed


,sum,count
Area,,
Dates,5,5
Duplicates,4,4
Grain,1,1
KPI,12,12
Keys,4,4
Read-only,1,1
Relationships,6,6
Risk,9,9
Rules,8,8
